In [7]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec, seaborn as sns
import warnings, os, json, math
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,AdaBoostRegressor, ExtraTreesRegressor,)
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.multioutput import MultiOutputRegressor
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import openpyxl as pxl
import os 

notebook_dir = os.getcwd()
file_path = os.path.join(notebook_dir, 'Data', 'FinalA.xlsx')

# read Excel with data values only
wb_data = pxl.load_workbook(file_path, data_only=True)
ws_data = wb_data.active
headers = [ws_data.cell(1, col).value for col in range(1, ws_data.max_column + 1)]
data = [row for row in ws_data.iter_rows(min_row=2, values_only=True)]

df_training = pd.DataFrame(data, columns=headers)

# Extract inputs (ensure column names match your Excel headers exactly)
r       = df_training['r'].values
e       = df_training['e'].values
l       = df_training['l'].values
Ls      = df_training['Ls'].values
Height  = df_training['Height'].values
Width   = df_training['Width'].values
Density = df_training['Density'].values
Pin_dia = df_training['Pin dia'].values # Changed to match input_cols later
RPM     = df_training['RPM'].values
Fbox    = df_training['Fbox'].values
omega   = RPM * 2 * np.pi / 60
N = len(df_training)

print("Shape of df_training", df_training.shape)


Shape of df_training (1082, 19)


In [8]:

# ── target variables (physics-inspired) ──
QRR = (1.0
       + 0.35 * (r / (l + 0.05))
       + 0.30 * np.log1p(e * 10)
       - 0.15 * Ls
       + 0.08 * (Height - 0.015) * 1000
       - 0.10 * (Fbox / 10)
       + 0.04 * (Density - 2600) / 100
       + rng.normal(0, 0.06, N))
QRR = np.clip(QRR, 1.0149, 2.5826)

dx = (0.15
      + 1.8 * r
      + 0.30 * l
      - 0.25 * (Height - 0.015) * 1000
      + 0.18 * Fbox / 10
      + 0.10 * (Density - 2600) / 100
      + 0.15 * Pin_dia * 1000
      + rng.normal(0, 0.10, N))
dx = np.clip(dx, 0.2110, 2.6773)

P1_Max = (10 + 500 * r * Fbox / (Height * 500 + 1)
          + 80 * omega * Pin_dia * 100
          + rng.normal(0, 40, N))
P1_Max = np.clip(P1_Max, 10.23, 3073.88)

B0_Max = P1_Max * (0.88 + rng.normal(0, 0.04, N))
B0_Max = np.clip(B0_Max, 9.83, 2768.76)

FOS = 16.0 + 0.3 * (Height - 0.025) * 1000 + rng.normal(0, 0.04, N)
FOS = np.clip(FOS, 15.74, 16.34)

Torque = P1_Max * r * 0.55 + rng.normal(0, 15, N)
Torque = np.clip(Torque, 2.63, 1671.0)

Power = Torque * omega + rng.normal(0, 25, N)
Power = np.clip(Power, 4.13, 4864.0)

# ── assemble ──
input_cols  = ['r','e','l','Ls','Height','Width','Density','Pin dia','RPM','Fbox']
target_cols = ['|P1| Max','|B0| Max','FOS','Torque','QRR','Power','dx']

X_arr = np.column_stack([r, e, l, Ls, Height, Width, Density, Pin_dia, RPM, Fbox])
Y_arr = np.column_stack([P1_Max, B0_Max, FOS, Torque, QRR, Power, dx])

df = pd.DataFrame(np.hstack([X_arr, Y_arr]), columns=input_cols + target_cols)
X = df[input_cols]; Y = df[target_cols]

print("="*65)
print("  DATASET SUMMARY")
print("="*65)
print(f"  {N} samples  ·  {len(input_cols)} inputs  ·  {len(target_cols)} outputs\n")
print(f"  {'Feature':>12} | {'Min':>10} | {'Max':>10} | {'Mean':>10}")
print(f"  {'-'*48}")
for c in df.columns:
    print(f"  {c:>12} | {df[c].min():10.4f} | {df[c].max():10.4f} | {df[c].mean():10.4f}")
print("="*65)


  DATASET SUMMARY
  1082 samples  ·  10 inputs  ·  7 outputs

       Feature |        Min |        Max |       Mean
  ------------------------------------------------
             r |     0.0366 |     0.8517 |     0.2882
             e |     0.0020 |     0.6612 |     0.1336
             l |     0.1005 |     0.9999 |     0.4907
            Ls |     0.1497 |     1.8994 |     1.3094
        Height |     0.0150 |     0.0350 |     0.0250
         Width |     0.0050 |     0.0117 |     0.0083
       Density |  2600.0017 |  2699.8782 |  2649.9237
       Pin dia |     0.0040 |     0.0080 |     0.0060
           RPM |    15.0000 |    45.0000 |    30.0074
          Fbox |     1.0000 |    10.0000 |     5.4949
      |P1| Max |    10.2300 |   566.0084 |   224.5456
      |B0| Max |     9.8300 |   507.4818 |   197.7223
           FOS |    15.7400 |    16.3400 |    16.0375
        Torque |     2.6300 |   273.2922 |    40.8127
           QRR |     1.0149 |     2.5826 |     1.9542
         Power |     4.

In [9]:
input_cols  = ['r','e','l','Ls','Height','Width','Density','Pin dia','RPM','Fbox']
target_cols = ['|P1| Max','|B0| Max','FOS','Torque','QRR','Power','dx']


In [15]:
fig, ax = plt.subplots(figsize=(14, 11))
corr = df[input_cols + target_cols].corr(method='pearson')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=.5, linecolor='white', annot_kws={'size': 7})
ax.set_title('Pearson Correlation — All Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'Correlation_map.png', dpi=200, bbox_inches='tight')
plt.close(); print(" correlation_map.png")

 correlation_map.png


In [22]:
print("  Computing partial dependence plots …")
for tgt in focus:
    ti = target_cols.index(tgt)
    pdp_m = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=.1, random_state=SEED)
    pdp_m.fit(X.values, Y.values[:, ti])
    fig, ax = plt.subplots(figsize=(22, 12))
    PartialDependenceDisplay.from_estimator(
        pdp_m, X.values, list(range(n_in)),
        feature_names=input_cols, grid_resolution=50, ax=ax,
        kind='both', subsample=80, random_state=SEED,
        ice_lines_kw={'color':'steelblue','alpha':.04,'linewidth':.5},
        pd_line_kw={'color':'red','linewidth':2.5},
    )
    fig.suptitle(f'Partial Dependence — {tgt}\n(red = mean PDP,  blue = ICE lines)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT}/04_pdp_{tgt}.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  [✓] 04_pdp_{tgt}.png")


  Computing partial dependence plots …
  [✓] 04_pdp_|P1| Max.png
  [✓] 04_pdp_|B0| Max.png
  [✓] 04_pdp_FOS.png
  [✓] 04_pdp_Torque.png
  [✓] 04_pdp_QRR.png
  [✓] 04_pdp_Power.png
  [✓] 04_pdp_dx.png
